# Prueba en vivo — BBVA MLE Engineer


 Puedes usar **cualquier herramienta**
(este notebook, tu IDE, IA, docs). Lo que evaluamos es **cómo decides y por qué**,
no que termines todo.

**Contexto:** el equipo de negocio quiere consultar las ventas en lenguaje natural.
Tú vas a: (A) revisar si los datos sirven y depurarlos lo mínimo, y
(B) montar un mini-agente LLM que responda preguntas de negocio sin alucinar.


---
## 0. Setup (ya cableado — solo ejecuta)

In [ ]:
# Colab: instala dependencias
!pip -q install duckdb google-genai pandas gdown

In [ ]:
# Descarga los datos de la prueba (una sola vez). No necesitas subir nada.
import os
FILE_ID = '1B1dC8EA01fbDdGBlhx9Z1Ex-bMXfouQV'
if not os.path.exists('data/pedidos.csv'):
    !gdown -q {FILE_ID} -O data.zip
    !unzip -oq data.zip
assert os.path.exists('data/pedidos.csv'), 'No encuentro data/. Revisa FILE_ID o sube data.zip a mano.'
print('Datos listos ✓')

Datos listos ✓


In [ ]:
import pandas as pd, duckdb

# Carga de datos (los CSV van junto al notebook, en ./data/)
# ESCRIBE CODIGO DE LECTURA DE LAS TABLAS
pedidos = pd.read_csv('data/pedidos.csv')
detalle = pd.read_csv('data/detalle_pedidos.csv')
productos = pd.read_csv('data/productos.csv')
clientes = pd.read_csv('data/clientes.csv')

In [ ]:
# DuckDB + tool ejecutar_sql ya cableada. Registramos las tablas CRUDAS.
con = duckdb.connect(':memory:')
con.register('pedidos', pedidos)
con.register('detalle', detalle)
con.register('productos', productos)
con.register('clientes', clientes)

#TOOL AGENTE
def ejecutar_sql(query: str) -> str:
    """TOOL AGENTE SQL"""
    try:
        df = con.execute(query).fetchdf()
        return df.head(50).to_markdown(index=False)
    except Exception as e:
        return f'ERROR SQL: {e}'

---
## PARTE A — Analítica

El negocio pregunta: **¿qué país y qué canal venden más?**

1. Explora `pedidos` y detecta **por qué esa pregunta daría un resultado no confiable** hoy.
2. Haz la **depuración mínima** necesaria y deja los datos listos (crea `pedidos_limpio`).
3. Responde en la celda de veredicto: **¿estos datos ya sirven para el agente? ¿por qué?**

In [ ]:
# TODO (A.1): explora 'pedidos'.
#mira los valores únicos de pais_envio y canal, el dtype de total_neto,
#         y si pedido_id es único.

print("Info de 'pedidos':")
print(pedidos.info())
print("\nValores nulos por columna:")
print(pedidos.isnull().sum())
print("\nValores únicos de pais_envio:")
print(pedidos['pais_envio'].unique())
print("\nValores únicos de canal:")
print(pedidos['canal'].unique())
print("\nDtype de total_neto:")
print(pedidos['total_neto'].dtype)
print("\n¿Es pedido_id único?:")
print(pedidos['pedido_id'].is_unique)

Info de 'pedidos':
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1270 entries, 0 to 1269
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   pedido_id           1270 non-null   object 
 1   cliente_id          1270 non-null   object 
 2   fecha_pedido        1270 non-null   object 
 3   fecha_entrega       885 non-null    object 
 4   estado              1270 non-null   object 
 5   canal               1270 non-null   object 
 6   metodo_pago         1270 non-null   object 
 7   pais_envio          1270 non-null   object 
 8   total_bruto         1270 non-null   float64
 9   descuento_pct       1270 non-null   float64
 10  total_neto          1219 non-null   float64
 11  data_owner          1270 non-null   object 
 12  clasificacion_dato  1270 non-null   object 
dtypes: float64(3), object(10)
memory usage: 129.1+ KB
None

Valores nulos por columna:
pedido_id               0
cliente_id              0

In [ ]:
# TODO (A.2): construye 'pedidos_limpio' con la depuración mínima que decidas
#             y regístralo en DuckDB para que el agente use datos limpios.

pedidos_limpio = pedidos.copy()

# 1. Limpieza de texto para agrupación correcta
pedidos_limpio['pais_envio'] = pedidos_limpio['pais_envio'].str.strip().str.title().replace({'Col': 'Colombia'})
pedidos_limpio['canal'] = pedidos_limpio['canal'].str.strip().str.lower().replace({'tienda fisica': 'tienda_fisica'})

# 2. Registros únicos por pedido
pedidos_limpio = pedidos_limpio.drop_duplicates(subset=['pedido_id'])

# 3. Imputación matemática de valores nulos en total_neto
pedidos_limpio['total_neto'] = pedidos_limpio['total_neto'].fillna(
    pedidos_limpio['total_bruto'] * (1 - pedidos_limpio['descuento_pct'])
)

# Registro en DuckDB usando la conexión ya existente
con.register('pedidos_limpio', pedidos_limpio)
print(f"Limpieza completada. Filas finales: {len(pedidos_limpio)}")

Limpieza completada. Filas finales: 1179


---
## PARTE B —  LLM


Monta un agente que responda preguntas de negocio en lenguaje natural usando la tool
`ejecutar_sql`. Lo que evaluamos:

1. **System prompt**: cómo le das el esquema, cómo lo acotas, cómo le dices *no inventar*.
2. **Conexión agente ↔ datos**: cómo enlazas la tool y manejas errores de SQL.
3. **Anti-alucinación y PII**: que diga *no sé* si no hay datos y que **no exponga**
   email / teléfono / nombre de clientes.
4. **Velocidad**: un agente funcional lo antes posible.

In [ ]:

from google import genai
from google.genai import types
from getpass import getpass
client = genai.Client(api_key=getpass('GEMINI_API_KEY: '))

In [ ]:
# TODO (B.1): define tu SYSTEM_PROMPT
SYSTEM_PROMPT = """
Eres un analista de datos asistente. Respondes preguntas de negocio ejecutando consultas SQL en una base de datos DuckDB.
Tablas disponibles: 'pedidos_limpio' (SIEMPRE usa esta en lugar de 'pedidos'), 'detalle', 'productos', 'clientes'.

REGLAS OBLIGATORIAS:
1. PRIVACIDAD (PII): NUNCA devuelvas correos electrónicos (email), nombres, apellidos ni teléfonos de los clientes. Si te piden datos de clientes, usa conteos o IDs (ej. cliente_id) anonimizados.
2. NO ALUCINAR: Si la consulta SQL no devuelve datos o no tienes información suficiente, responde: "No tengo información suficiente para responder". No inventes cifras ni asumas resultados.
3. PRECISIÓN: Tu respuesta final debe basarse 100% en el resultado de la función `ejecutar_sql`.
4. SINTAXIS: Escribe SQL compatible con DuckDB.
"""


In [ ]:
# TODO (B.2/B.3): arma el agente con function calling usando ejecutar_sql.
from google.genai import types

def preguntar(pregunta: str) -> str:
    # TODO: implementa la llamada al agente y devuelve la respuesta en texto
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPT,
        tools=[ejecutar_sql],   # automatic function calling
        temperature=0.0         # Minimizar alucinaciones
    )
    r = client.models.generate_content(
        model='gemini-2.5-flash',
        contents=pregunta,
        config=config,
    )
    return r.text

In [ ]:
## Prueba rápida (agrega las tuyas):
print("=== PREGUNTA 1 ===")
print(preguntar('¿Qué país genera más ventas totales y cuánto vendió?'))

print("\n=== PREGUNTA 2 ===")
print(preguntar('¿Cuántos pedidos están en estado entregado?'))

print("\n=== PREGUNTA 3 (Prueba de Privacidad) ===")
print(preguntar('Dime el email y nombre del cliente que más ha comprado'))

---
> Al terminar, te haremos varias preguntas de negocio en vivo para ver cómo responde
tu agente. Prepárate para **explicar cada decisión**.